# 05 Hauptexperiment: neun Embedder, BM25 und BM25-de auf GermanQuAD

## Schritt 0: Überblick, Lesehilfe und Glossar

**Was?** Dieses Notebook beantwortet die Forschungsfrage der Arbeit: Welche von neun Embedding-Modellen finden zu einer deutschen Frage die passende Passage besser als die klassische Stichwortsuche BM25, und wie unterscheiden sich die Modelle untereinander? Datengrundlage ist der Testsplit von GermanQuAD-Retrieval (2.204 Fragen, 474 Passagen, je Frage genau eine passende Passage).

**Warum in Schritten?** Das Notebook trennt das langsame Berechnen der Vektoren von allem Übrigen. Ranking, Kennzahlen und Tests laufen aus gespeicherten Vektoren in Sekunden bis Minuten und lassen sich beliebig oft wiederholen. Jeder Schritt beginnt mit „Was? Warum? Wie?“.

**Wie?** Jeder Schritt lädt gespeicherte Ergebnisse aus `code/output/` und zeigt sie an. Zwei Schalter in der Konfigurationszelle unten steuern, ob ein Schritt vorher neu berechnet wird.

| Schritt | Inhalt | Laufzeit bei Neuberechnung | Schalter |
|---|---|---|---|
| 1 | Daten laden (Fragen, Passagen, Zuordnung) | Sekunden | – |
| 2 | Encoding: Texte werden zu Vektoren (Cache) | Minuten bis Stunden je Modell (CPU) | `ENCODE` |
| 3 | Retrieval/Ranking und Kennzahlen aus dem Cache | Sekunden | `SCORE` |
| 4 | Signifikanztests (103 Tests in 22 Familien, sechs konfirmatorisch) | wenige Minuten | `SCORE` |
| 5 | Sensitivität: Passagen auf 512 Tokens gekürzt | (Teil von 2 und 3) | `ENCODE`, `SCORE` |
| 6 | BM25 Standard und BM25 mit deutscher Vorverarbeitung | (Teil von 3 und 4) | `SCORE` |
| 7 | Drei explorativ getestete Modelle (Arctic, Qwen3, jina-de) | (Teil von 2 bis 4) | `ENCODE`, `SCORE` |
| 8 | Zusammenfassung, Laufzeiten und Konsistenzprüfung | Sekunden | – |

**Schalter.** Beide Schalter stehen standardmäßig auf `False`. Dann lädt das Notebook nur die gespeicherten Ergebnisse (ohne Netz, GPU und API, wenige Sekunden).

| `ENCODE` | `SCORE` | Wirkung |
|---|---|---|
| False | False | Nur laden (Standard) |
| True | False | Vektoren berechnen und im Cache speichern (Modell-Download, Stunden) |
| False | True | Ranking, Kennzahlen und Tests aus dem Cache neu berechnen (Cache muss vorhanden sein) |
| True | True | Beides nacheinander |

**Woher kommen die Daten?** Die normalisierten Dateien `queries/docs/qrels.normalized.jsonl` liegen fertig in `code/output/germanquad/`. Die Notebooks 01 (Download) und 02 (Parsen und Normalisieren) sind für dieses Notebook nicht nötig; sie werden nur gebraucht, um diese Dateien aus den Rohdaten neu zu erzeugen.

### Glossar in Alltagssprache

- **Embedding-Modell.** Ein Programm, das einen Text in eine lange Zahlenliste (einen *Vektor*, z. B. 1.024 Zahlen) übersetzt. Texte mit ähnlicher Bedeutung erhalten ähnliche Zahlenlisten.
- **Training und Encoding.** *Training* heißt: Das Modell lernt aus riesigen Textmengen (Wochen auf Großrechnern). Das ist bei allen neun Modellen bereits geschehen; sie sind *vortrainiert*, wir laden die fertige, per Versionsnummer festgelegte Fassung. *Encoding* heißt: Wir geben unsere 474 Passagen und 2.204 Fragen in das fertige Modell und lesen die Vektoren ab. Dabei ändert sich am Modell nichts. Das Encoding ist der langsame Teil (auf CPU Minuten bis Stunden je Modell).
- **Cache.** Die einmal berechneten Vektoren werden als `.npy`-Dateien gespeichert (`output/harness/cache/`, nicht in Git). Ranking und Tests nutzen nur den Cache und brauchen das Modell nicht mehr.
- **Retrieval per Kosinus.** Zu einer Frage wird der Kosinus-Ähnlichkeit (hier ein Skalarprodukt normierter Vektoren) zu allen 474 Passagenvektoren berechnet. Sortiert nach Ähnlichkeit ergibt sich eine Rangliste der Passagen. Das dauert Sekunden.
- **Gold-Passage.** Die Passage, zu der die Frage im Datensatz gestellt wurde. Sie gilt als die richtige Antwortstelle; je Frage gibt es genau eine.
- **Retrieval-Qualität.** Wie weit oben die Gold-Passage in der Rangliste steht.
- **MRR@10.** Für jede Frage der Kehrwert des Rangs der Gold-Passage (Rang 1 gibt 1, Rang 2 gibt 0,5, Rang 3 gibt 0,33 …; außerhalb der Top 10 gibt es 0), gemittelt über alle Fragen. Hauptkennzahl der Arbeit.
- **Success@k.** Anteil der Fragen, bei denen die Gold-Passage unter den ersten *k* Treffern liegt (k = 1, 5, 10).
- **BM25.** Klassische Stichwortsuche; die Vergleichsbasis (Baseline). *BM25-de* ist dieselbe Suche mit deutscher Vorverarbeitung (Stoppwörter entfernt, Wortstämme gebildet).
- **Native Eingabelänge.** Jedes Modell verarbeitet Text nur bis zu einer Höchstlänge in Tokens (Wortstücken). *Nativ* heißt: Standardlänge des Modells (e5-large: 512, bge-m3, gte und jina-v3: bis 8.192). 143 von 474 Passagen sind länger als 512 Tokens.
- **512-Token-Sensitivität.** Dieselben Modelle (bge-m3, gte, jina-v3) werden zusätzlich mit auf 512 Tokens gekürzten Passagen gerechnet. So zeigt sich, ob Unterschiede zu e5-large an der Kürzung hängen.
- **Konfirmatorischer Test.** Die vorab festgelegte Hauptfrage (F1): Liegt jedes der sechs Modelle bei MRR@10 über BM25? Das sind genau sechs Tests, und nur sie tragen eine Aussage der Arbeit. Gewählt, weil F1 die Forschungsfrage ist und die Festlegung vor den Läufen verhindert, dass nachträglich die günstigsten Vergleiche ausgesucht werden.
- **Holm-Korrektur.** Bei sechs gleichzeitigen Tests steigt die Chance auf Zufallsfunde. Die Holm-Korrektur rechnet die p-Werte so um, dass die Gesamtfehlerquote bei 5 % bleibt.
- **Cluster nach Passage.** Mehrere Fragen gehören zur selben Passage (2.204 Fragen, 474 Passagen) und sind daher nicht unabhängig. Die Tests zählen deshalb 474 Cluster statt 2.204 Fragen; sonst wären die Ergebnisse zu optimistisch.
- **Explorative Tests.** Zusatzvergleiche (Modelle untereinander, Success@k, 512 Tokens, BM25-de, die drei explorativ getesteten Modelle). Sie liefern Hinweise für die Diskussion; Holm gilt nur innerhalb ihrer jeweiligen Gruppe.
- **n≠0.** Anzahl der Cluster mit Unterschied zwischen zwei Retrievern. Unter 20 gilt ein Vergleich als „nicht aussagekräftig“ (Deckeneffekt: beide Retriever sind fast immer richtig).

### Versuchsaufbau (Kurzfassung)

- **Datensatz:** `mteb/germanquad-retrieval` (Revision `9af36714ebd9bba5235c4b9862779adaddbfae52`) und `mteb/germanquad-retrieval-qrels` (Revision `ed232776ce5d736028c1da18e81ab7048969f369`, Split test), normalisiert in `code/output/germanquad/`. Retrieval auf Dokumentebene mit Kosinus-Ähnlichkeit.
- **Retriever:** BM25 (`rank_bm25`, k1 = 1,5, b = 0,75) und neun Embedder aus `pylib/embedders.py`: multilingual-e5-large, bge-m3, gte-multilingual-base, jina-embeddings-v3, text-embedding-3-small, text-embedding-3-large (konfirmatorische Familie, sechs Modelle) sowie Qwen3-Embedding-0.6B, snowflake-arctic-embed-l-v2.0, jina-embeddings-v2-base-de (explorative Familie, drei Modelle).
- **Kennzahlen:** Recall@k, Success@k, MRR@k, Precision@k für k = 1, 5, 10; Primärmetrik MRR@10.
- **Signifikanz:** gepaarter Vorzeichenwechsel-Randomisierungstest auf Cluster-Ebene (B = 100.000, Seed 42, exakt bei n≠0 ≤ 16), 95-%-Cluster-Bootstrap-CI (B = 10.000), Holm-Korrektur (α = 0,05). Konfirmatorisch: jeder Embedder gegen BM25 auf MRR@10 (6 Tests). Verbindliche Methode: `docs/evaluation_protocol.md`.

### Konfiguration

**Was?** Schalter, Modelllisten, Hilfsfunktionen und Plot-Stil für alle folgenden Schritte. **Warum?** Alles Gemeinsame steht an einer Stelle. **Wie?** `ENCODE` und `SCORE` schalten die Neuberechnung je Schritt ein; `ONLY` beschränkt sie auf einzelne Modellschlüssel (z. B. `["bge-m3"]`). Jeder Lauf eines Skripts wird mit Dauer in `output/harness/run_log.jsonl` protokolliert.

In [ ]:
import json, os, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

ENCODE = False  # True: Vektoren berechnen und im Cache speichern (Modell-Download, Stunden auf CPU)
SCORE = False   # True: Ranking, Kennzahlen und Tests aus dem Cache neu berechnen (überschreibt Ergebnisdateien)
ONLY = None     # z. B. ["bge-m3"]: Encoding und Ranking auf diese Modellschlüssel beschränken

CODE = Path.cwd() if (Path.cwd() / "embed_eval.py").exists() else Path.cwd() / "code"
HARNESS = CODE / "output" / "harness"
sys.path.insert(0, str(CODE))
sys.path.insert(0, str(CODE / "pylib"))
import compare_models as cm
from local_dataset_io import load_jsonl
from retrieval_metrics import build_gold
from significance import cluster_sums, holm, make_clusters

MODELS = {"e5-large": "multilingual-e5-large", "bge-m3": "bge-m3", "gte-multilingual-base": "gte-multilingual-base",
          "jina-v3": "jina-embeddings-v3", "openai-3-small": "text-embedding-3-small",
          "openai-3-large": "text-embedding-3-large"}
LEN512 = ["bge-m3", "gte-multilingual-base", "jina-v3"]
EXT = {"qwen3-embedding-0.6b": "Qwen3-Embedding-0.6B", "arctic-embed-l-v2": "Arctic-Embed-L-v2.0",
       "jina-v2-base-de": "jina-embeddings-v2-base-de"}
LABEL = {"bm25": "BM25", "bm25_de": "BM25-de", **MODELS, **EXT}
ALL = [*MODELS, *EXT]  # alle neun Embedder (sechs konfirmatorisch, drei explorativ)

plt.rcParams.update({"figure.figsize": (7, 4), "font.size": 10, "axes.titlesize": 11, "axes.grid": True, "grid.alpha": 0.3})
pd.options.display.float_format = "{:.4f}".format
TIMINGS = []  # (Schritt, Sekunden) der Läufe in dieser Sitzung


def keep(keys):
    return [k for k in keys if ONLY is None or k in ONLY]


def embed_cmd(model, stage, out_dir=None, max_len=None):
    cmd = [sys.executable, str(CODE / "embed_eval.py"), "--dataset-dir", "output/germanquad", "--model", model, "--stage", stage]
    if max_len:
        cmd += ["--max-seq-length", str(max_len)]
    return (f"{model} ({stage}" + (f", {max_len} Tokens" if max_len else "") + ")", cmd + (["--out-dir", str(out_dir)] if out_dir else []))


def script_cmd(name, *args):
    return (name, [sys.executable, str(CODE / name), *args])


def run_steps(steps, active, switch):
    """Führt (Name, Befehl)-Paare nacheinander aus, wenn der Schalter aktiv ist; sonst wird nur aufgelistet."""
    if not active:
        print(f"{switch} = False: geladen werden die gespeicherten Ergebnisse. Mit {switch} = True liefen nacheinander:")
        for name, _ in steps:
            print("  -", name)
        return
    for name, cmd in steps:
        t0 = time.time()
        r = subprocess.run(cmd, cwd=CODE, capture_output=True, text=True)
        sec = time.time() - t0
        if r.returncode:
            print(r.stderr[-2000:])
            raise RuntimeError(f"Schritt fehlgeschlagen: {name}")
        TIMINGS.append((name, round(sec, 1)))
        with (HARNESS / "run_log.jsonl").open("a", encoding="utf-8") as f:
            f.write(json.dumps({"step": name, "seconds": round(sec, 1), "switch": switch,
                                "timestamp_utc": datetime.now(timezone.utc).isoformat()[:19]}, ensure_ascii=False) + "\n")
        print(f"{name}: fertig in {sec:.0f} s")


def load_json(path):
    return json.loads(Path(path).read_text())


def encode_record(run_dir):
    """(Sekunden, Quelle) der echten Encodierung eines Laufs, gelesen aus den gespeicherten Dateien."""
    enc, met = run_dir / "encode_germanquad.json", run_dir / "metrics_germanquad.json"
    if enc.exists():
        return json.loads(enc.read_text())["environment"]["encode_seconds"], "encode_germanquad.json"
    if met.exists():
        e = json.loads(met.read_text())["environment"]
        if not (e.get("cache_hit_docs") and e.get("cache_hit_queries")) or "encode_source" in e:
            return e["encode_seconds"], "metrics_germanquad.json"
        return None, "Cache gelesen, Dauer nicht gespeichert"
    return None, "Ergebnisdatei fehlt"


def encode_timestamp(run_dir):
    """UTC-Zeitstempel der Encodierung (encode_germanquad.json, sonst metrics_germanquad.json)."""
    f = run_dir / ("encode_germanquad.json" if (run_dir / "encode_germanquad.json").exists() else "metrics_germanquad.json")
    return load_json(f)["environment"]["timestamp_utc"][:19]


def metrics_of(r):
    return {"MRR@10": cm.rr10(r).mean(), "Success@1": cm.succ(r, 1).mean(),
            "Success@5": cm.succ(r, 5).mean(), "Success@10": cm.succ(r, 10).mean()}


def test_table(df, names):
    """Testzeilen als Tabelle: Differenz, CI, p roh, p Holm, n≠0."""
    t = pd.DataFrame({"Vergleich": names, "Differenz": df.diff_b_minus_a.to_numpy(),
                      "CI95 unten": df.ci95_cluster_bootstrap.str[0].to_numpy(), "CI95 oben": df.ci95_cluster_bootstrap.str[1].to_numpy(),
                      "p roh": df.p_cluster_signflip.to_numpy(), "p Holm": df.p_holm.to_numpy(),
                      "n≠0": df.n_nonzero_clusters.to_numpy(), "nicht aussagekräftig": df.not_informative.to_numpy()})
    return t.style.format({"p roh": "{:.2e}", "p Holm": "{:.2e}"}, precision=4).hide(axis="index")


print("code dir:", CODE, "| ENCODE =", ENCODE, "| SCORE =", SCORE)

## Schritt 1: Daten laden

**Was?** Das Notebook liest Fragen (`queries`), Passagen (`docs`) und die Zuordnung Frage → Gold-Passage (`qrels`) aus `code/output/germanquad/`. **Warum?** Alle Retriever werden an denselben 2.204 Fragen und 474 Passagen gemessen; die Zuordnung liefert die richtige Antwort, gegen die jedes Ranking geprüft wird. **Wie?** Aus den `qrels` entsteht je Frage die Gold-Passage; sie dient zugleich als Cluster für die Tests (Schritt 4). Ergebnis: `qids`, `gold`, `clusters` für alle weiteren Schritte.

In [ ]:
queries, docs, qrels = (load_jsonl(CODE / "output" / "germanquad" / f"{x}.normalized.jsonl") for x in ("queries", "docs", "qrels"))
qids = [str(q["query_id"]) for q in queries]
gold = build_gold(qrels, {str(d["doc_id"]) for d in docs}, 1)
clusters = [sorted(gold[q])[0] for q in qids]
print(f"{len(qids)} Fragen, {len(docs)} Passagen, {len(set(clusters))} Gold-Passagen (Cluster), je Frage {set(len(gold[q]) for q in qids)} relevante Passage(n)")
ex = queries[0]; doc = next(d for d in docs if str(d["doc_id"]) == sorted(gold[str(ex["query_id"])])[0])
print("Beispielfrage:", ex["query_text"]); print("Gold-Passage (Anfang):", doc["text"][:200].replace("\n", " "), "...")

## Schritt 2: Encoding (Modelle → Vektoren, Cache, Laufzeit)

**Was?** Jedes Modell übersetzt alle 474 Passagen und 2.204 Fragen in Vektoren. Die Modelle sind vortrainiert; hier wird nichts trainiert, nur kodiert. **Warum?** Das ist der rechenintensive Teil (auf CPU Minuten bis Stunden je Modell, die OpenAI-Modelle laufen über die API). Die Vektoren werden einmal berechnet und als `.npy` im Cache gespeichert. **Wie?** Mit `ENCODE = True` startet die Zelle `embed_eval.py --stage encode` für alle neun Modelle (nativ) und für drei Modelle zusätzlich mit `--max-seq-length 512`. Die OpenAI-Modelle laufen nur, wenn `EMBED_OPENAI_API_KEY` gesetzt ist. Benötigt werden die Pakete aus `requirements-embed.txt`. Die Tabelle darunter zeigt die Dauer der echten Encodierung aus den gespeicherten Dateien (`encode_germanquad.json`, sonst `metrics_germanquad.json`); fehlt die Dauer, steht `–`.

In [ ]:
main_keys = [m for m in keep(ALL) if not (m.startswith("openai") and not os.environ.get("EMBED_OPENAI_API_KEY"))]
if ENCODE and set(keep(ALL)) - set(main_keys):
    print("übersprungen (EMBED_OPENAI_API_KEY fehlt):", sorted(set(keep(ALL)) - set(main_keys)))
steps = [embed_cmd(m, "encode") for m in main_keys] + [embed_cmd(m, "encode", max_len=512) for m in keep(LEN512)]
run_steps(steps, ENCODE, "ENCODE")

In [ ]:
runs = [(m, LABEL[m], "nativ", HARNESS / m) for m in ALL] + [(f"{m}__len512", LABEL[m], "512 Tokens", HARNESS / f"{m}__len512") for m in LEN512]
rows = []
for key, label, mode, d in runs:
    sec, src = encode_record(d)
    dim = load_json(d / "metrics_germanquad.json")["overview"]["embedding_dim"] if (d / "metrics_germanquad.json").exists() else None
    rows.append({"Lauf": key, "Modell": label, "Eingabelänge": mode, "Vektordimension": dim, "Encodierung (s)": sec, "Quelle": src})
enc_tab = pd.DataFrame(rows)
display(enc_tab.style.format({"Encodierung (s)": "{:.0f}"}, na_rep="–").hide(axis="index"))
tot = enc_tab["Encodierung (s)"].sum()
print(f"Summe der gespeicherten Encodierzeiten: {tot:.0f} s (etwa {tot / 3600:.1f} h, 4 CPU-Kerne laut metrics-Dateien)")
ax = enc_tab.dropna(subset=["Encodierung (s)"]).set_index("Lauf")["Encodierung (s)"].sort_values().plot.barh()
ax.set_xlabel("Sekunden"); ax.set_title("Encoding-Dauer je Lauf (aus gespeicherten Dateien)"); plt.tight_layout(); plt.show()

## Schritt 3: Retrieval/Ranking und Kennzahlen

**Was?** Aus den gespeicherten Vektoren wird für jede Frage die Rangliste aller 474 Passagen (Kosinus) gebildet und daraus der Rang der Gold-Passage bestimmt. BM25 Standard und BM25-de werden hier mitgerankt (Details in Schritt 6). **Warum?** Der Rang der Gold-Passage ist die Grundlage aller Kennzahlen (MRR@10, Success@k) und Tests. Das Ranking dauert Sekunden und ist aus dem Cache beliebig wiederholbar. **Wie?** Mit `SCORE = True` startet die Zelle `embed_eval.py --stage score` je Modell (liest nur den Cache, lädt kein Modell) sowie `scripts/run_bm25.py` und `bm25_german.py`. Ergebnis je Lauf: `metrics_germanquad.json` (Kennzahlen) und `per_query_germanquad.csv` (Rang der Gold-Passage je Frage); darunter Tabelle 2.

In [ ]:
steps = [("BM25 Standard", [sys.executable, str(CODE / "scripts" / "run_bm25.py")]), script_cmd("bm25_german.py")]
steps += [embed_cmd(m, "score") for m in keep(ALL)] + [embed_cmd(m, "score", max_len=512) for m in keep(LEN512)]
run_steps(steps, SCORE, "SCORE")

In [ ]:
NATIVE = {m: load_json(HARNESS / m / "metrics_germanquad.json") for m in ALL}
S512 = {m: load_json(HARNESS / f"{m}__len512" / "metrics_germanquad.json") for m in LEN512}
RANKS = {m: cm.load_ranks(HARNESS / m / "per_query_germanquad.csv", qids) for m in ALL}
RANKS["bm25"] = cm.load_ranks(cm.BM25_CSV, qids)
RANKS["bm25_de"] = cm.load_ranks(cm.BM25_DE_CSV, qids)
R512 = {m: cm.load_ranks(HARNESS / f"{m}__len512" / "per_query_germanquad.csv", qids) for m in LEN512}
score_s = {LABEL[m]: d["environment"].get("score_seconds") for m, d in NATIVE.items()}
print("Ranking-Dauer je Modell (s, gespeichert; – = Lauf vor Einführung des Feldes):", {k: v if v is not None else "–" for k, v in score_s.items()})

### Tabelle 2: Retrievalqualität (native Eingabelänge)

In [ ]:
t2 = pd.DataFrame({LABEL[m]: metrics_of(RANKS[m]) for m in ["bm25", *MODELS]}).T
display(t2)

## Schritt 4: Signifikanztests

**Was?** Die Tests prüfen, ob die Unterschiede in MRR@10 zwischen den Retrievern größer sind, als der Zufall bei dieser Fragenauswahl erwarten lässt. Die **konfirmatorische** Familie enthält sechs Tests: jedes der sechs zuerst festgelegten Modelle gegen BM25, Holm-korrigiert. Alle übrigen 97 Tests (Paare der sechs Modelle, Success@k, BM25-de, drei weitere Modelle) sind **explorativ** und werden je Familie (gleiche Referenz und Kennzahl) korrigiert. **Warum?** Die Forschungsfrage F1 lautet „die sechs zuerst festgelegten Modelle gegen BM25“; nur dafür wurden Test und Korrektur vorab festgelegt. Die Cluster nach Passage berücksichtigen, dass mehrere Fragen zur selben Passage gehören. **Wie?** Mit `SCORE = True` rechnet `compare_models.py` in einem Lauf alle 103 Tests (Seed 42) und `make_results.py` baut daraus Tabellen, Abbildung 1, die Paarmatrix und das Ablaufschema (ohne den Vergleich erneut zu rechnen). Darunter stehen die Tabellen; Schritt 8 prüft die Kennzahlen gegen die Ränge.

In [ ]:
run_steps([script_cmd("compare_models.py"), script_cmd("make_results.py")], SCORE, "SCORE")
CMP = load_json(HARNESS / "comparison_germanquad.json")
TESTS = pd.DataFrame(CMP["tests"])
m = CMP["meta"]
print(f'Vergleich: Seed {m["seed"]}, B_perm {m["B_permutation"]}, B_boot {m["B_bootstrap"]}, Schwelle n≠0 < {m["threshold_not_informative"]}, Stand {m["timestamp_utc"][:19]} UTC')

### Tabelle 3: Konfirmatorische Tests (Embedder gegen BM25, MRR@10, Holm über 6 Tests)

In [ ]:
conf = TESTS[TESTS.family.str.startswith("confirmatory")].copy()
display(test_table(conf, (conf.b.map(LABEL) + " - BM25").to_numpy()))

### Explorative Paarvergleiche der Embedder (MRR@10, Holm innerhalb der Gruppe)

In [ ]:
pairs = TESTS[TESTS.family == "explorative: embedder pairs, MRR@10"]
display(test_table(pairs, (pairs.b.map(LABEL) + " - " + pairs.a.map(LABEL)).to_numpy()))

### Abbildung 1: MRR@10 mit 95-%-Cluster-Bootstrap-CI

In [ ]:
display(Image(filename=str(HARNESS / "results" / "fig_mrr10_ci.png")))

## Schritt 5: Sensitivität 512 Tokens

**Was?** bge-m3, gte-multilingual-base und jina-v3 werden zusätzlich mit auf 512 Tokens gekürzten Passagen gerechnet und mit ihrem nativen Lauf verglichen. **Warum?** e5-large verarbeitet höchstens 512 Tokens, die drei anderen bis 8.192. Da 143 von 474 Passagen länger als 512 Tokens sind, zeigt der Vergleich, ob Unterschiede zu e5-large an der Kürzung hängen. Die Analyse ist explorativ (ohne Holm). **Wie?** Encoding (`--max-seq-length 512`) und Ranking laufen in Schritt 2 und 3 mit; hier stehen die Kennzahlen nativ gegen 512. Die Cluster-Bootstrap-CI der Differenz stehen in `output/harness/results/table_len512.md`.

In [ ]:
rows = []
for m in LEN512:
    a, b = metrics_of(RANKS[m]), metrics_of(R512[m])
    rows.append({"Modell": LABEL[m], "MRR@10 nativ": a["MRR@10"], "MRR@10 512": b["MRR@10"], "Differenz": b["MRR@10"] - a["MRR@10"],
                 "Success@1 nativ": a["Success@1"], "Success@1 512": b["Success@1"],
                 "Eff. max_seq_length nativ": NATIVE[m]["experiment"]["max_seq_length_effective"],
                 "Eff. 512-Lauf": S512[m]["experiment"]["max_seq_length_effective"]})
display(pd.DataFrame(rows).style.format(precision=4).hide(axis="index"))
tl = load_json(HARNESS / "token_lengths.json")
print("Passagen > 512 Tokens (je Tokenizer):", {k: v["n_over_512"] for k, v in tl["models"].items()})

## Schritt 6: BM25 Standard und BM25 mit deutscher Vorverarbeitung

**Was?** BM25 sucht über übereinstimmende Wörter. *BM25 Standard* zerlegt den Text nur in Kleinbuchstaben-Wörter und ist die Vergleichsbasis der konfirmatorischen Tests. *BM25-de* entfernt zusätzlich deutsche Stoppwörter und reduziert Wörter auf ihren Stamm (Snowball). **Warum?** BM25-de zeigt, wie viel vom Abstand der Modelle zu BM25 auf fehlende Vorverarbeitung zurückgeht. Die Auswertung ist explorativ; die konfirmatorische Gruppe bleibt unverändert. **Wie?** `bm25_german.py` rankt BM25-de in Schritt 3, die Tests gegen BM25 Standard und gegen die Embedder stammen aus dem Lauf von `compare_models.py` in Schritt 4 (Holm je Familie). Benötigt werden `stopwordsiso` und `snowballstemmer` (`requirements-embed.txt`).

In [ ]:
BM = load_json(CODE / "output" / "bm25_de" / "bm25_results.json")["experiment"]
print("BM25-de Vorverarbeitung:", "; ".join(BM["preprocessing"]["steps"]), f'({BM["preprocessing"]["n_stopwords"]} Stoppwörter)')
display(pd.DataFrame({LABEL[m]: metrics_of(RANKS[m]) for m in ["bm25", "bm25_de"]}).T)

In [ ]:
DE = TESTS
g = DE[DE.family.str.contains("BM25-de vs BM25 standard")]
display(Markdown("**BM25-de gegen BM25 Standard** (4 Kennzahlen, Differenz = BM25-de minus BM25)"))
display(test_table(g, g.metric.to_numpy()))
g = DE[DE.family == "explorative: embedder vs BM25-de, MRR@10"]
display(Markdown("**Embedder gegen BM25-de** (MRR@10, Differenz = Embedder minus BM25-de)"))
display(test_table(g, (g.b.map(LABEL) + " - BM25-de").to_numpy()))

## Schritt 7: Drei explorativ getestete Modelle (Arctic, Qwen3, jina-de)

**Was?** Drei zusätzliche Modelle (Arctic-Embed-L-v2.0, Qwen3-Embedding-0.6B, jina-embeddings-v2-base-de) werden wie die übrigen Modelle encodiert, gerankt und mit BM25, BM25-de und bge-m3 verglichen. **Warum?** Gemeinsam mit den sechs Modellen aus Schritt 3 bis 6 ergeben sie die neun Modelle der Arbeit. Ihre Vergleiche bilden eigene explorative Familien; die konfirmatorische Familie bleibt bewusst bei sechs Modellen fester Größe. **Wie?** Encoding (Schritt 2), Ranking (Schritt 3) und Tests (Schritt 4) laufen gemeinsam mit den übrigen Modellen; die Ergebnisse liegen wie dort unter `output/harness/<modell>/`.

In [ ]:
ALL_RANKS = RANKS
rows = []
for m in EXT:
    sec, src = encode_record(HARNESS / m)
    rows.append({"Modell": LABEL[m], **metrics_of(RANKS[m]), "Vektordimension": NATIVE[m]["overview"]["embedding_dim"], "Encodierung (s)": sec})
for m in ["bm25", "bm25_de", "bge-m3"]:
    rows.append({"Modell": LABEL[m] + " (Referenz)", **metrics_of(ALL_RANKS[m])})
display(pd.DataFrame(rows).style.format(precision=4, na_rep="–").hide(axis="index"))

In [ ]:
EX = TESTS[TESTS.family.str.contains("new model")]
g = EX[EX.metric == "MRR@10"]
display(Markdown("**MRR@10 der Erweiterungsmodelle gegen Referenzen** (Differenz = Erweiterungsmodell minus Referenz; Holm je Referenz)"))
display(test_table(g, (g.b.map(LABEL) + " - " + g.a.map(LABEL)).to_numpy()))

## Schritt 8: Zusammenfassung und Konsistenzprüfung

**Was?** Der Schritt fasst Laufzeiten und Rohbefunde zusammen und prüft, ob die gespeicherten Testergebnisse zu den Rängen passen. **Warum?** Die Prüfung schützt vor veralteten oder vermischten Ergebnisdateien. **Wie?** Die Kennzahlen werden aus den Per-Query-Rängen neu berechnet und gegen `comparison_germanquad.json` und die `metrics_germanquad.json` der Modelle gehalten: Mittelwerte, Differenzen, n≠0, Holm-Korrektur und Deckeneffekt-Flag. Die p-Werte und CI werden nicht neu simuliert; das leistet `compare_models.py`.

### Laufzeiten je Schritt

In [ ]:
rows = [{"Schritt": "Encoding (Summe neun Modelle und 512-Läufe)", "Sekunden": enc_tab["Encodierung (s)"].sum(), "Quelle": "encode_germanquad.json / metrics_germanquad.json"}]
log = HARNESS / "run_log.jsonl"
if log.exists():
    last = {}
    for line in log.read_text(encoding="utf-8").splitlines():
        r = json.loads(line); last[r["step"]] = r
    rows += [{"Schritt": f'{k} (letzter Lauf {v["timestamp_utc"]} UTC)', "Sekunden": v["seconds"], "Quelle": "run_log.jsonl"} for k, v in last.items()]
else:
    print("Kein run_log.jsonl: Ranking und Tests wurden noch nicht über dieses Notebook gelaufen; Dauern ohne gespeicherten Wert werden nicht angegeben.")
display(pd.DataFrame(rows).style.format({"Sekunden": "{:.0f}"}).hide(axis="index"))

### Konsistenzprüfung

In [ ]:
cinv, ncl, sizes = make_clusters(clusters)
assert len(qids) == 2204 and ncl == 474

def check_tests(cmp, thr):
    assert cmp["n_queries"] == len(qids) and cmp["n_clusters"] == ncl
    for t in cmp["tests"]:
        f, ra, rb = cm.METRICS[t["metric"]], ALL_RANKS[t["a"]], ALL_RANKS[t["b"]]
        da, db = f(ra), f(rb)
        assert abs(da.mean() - t["mean_a"]) < 1e-9 and abs(db.mean() - t["mean_b"]) < 1e-9, t
        assert abs((db - da).mean() - t["diff_b_minus_a"]) < 1e-9, t
        nz = int((np.abs(cluster_sums(db - da, cinv, ncl)) > 1e-12).sum())
        assert nz == t["n_nonzero_clusters"], (t["a"], t["b"], t["metric"], nz)
        assert t["not_informative"] == (nz < thr)
    df = pd.DataFrame(cmp["tests"])
    for fam, g in df.groupby("family"):
        size = 6 if fam.startswith("confirmatory") else None
        assert np.allclose(holm(g.p_cluster_signflip.tolist(), size), g.p_holm.to_numpy()), fam
    return len(df)

n_main = check_tests(CMP, CMP["meta"]["threshold_not_informative"])
assert n_main == 103 and TESTS.family.nunique() == 22
for m, d in NATIVE.items():
    v = metrics_of(ALL_RANKS[m]); mm = {x["k"]: x for x in d["metrics"]}
    assert abs(mm[10]["mrr_at_k"] - v["MRR@10"]) < 1e-9 and abs(mm[1]["success_at_k"] - v["Success@1"]) < 1e-9, m
    assert abs(mm[5]["success_at_k"] - v["Success@5"]) < 1e-9 and abs(mm[10]["success_at_k"] - v["Success@10"]) < 1e-9, m
for m, d in S512.items():
    assert abs({x["k"]: x for x in d["metrics"]}[10]["mrr_at_k"] - metrics_of(R512[m])["MRR@10"]) < 1e-9, m
assert len(conf) == 6 and (conf.a == "bm25").all()
print(f"Konsistenzprüfung bestanden: {n_main} Tests in {TESTS.family.nunique()} Familien (6 konfirmatorisch), "
      f"{len(NATIVE)} Modelle, {len(S512)} 512-Läufe.")

### Umgebung und Provenienz

In [ ]:
rows = []
for m, d in NATIVE.items():
    e, env = d["experiment"], d["environment"]
    api = e.get("openai_api", {})
    rows.append({"Modell": e["model_name"], "Revision / API-Modell": e.get("model_revision") or api.get("returned_models", ["?"])[0],
                 "Remote-Code-Revision": e.get("code_revision_loaded", ""), "max_seq_length effektiv": e.get("max_seq_length_effective", "API"),
                 "Präfixe/Tasks": (e["query_prefix"] + "|" + e["doc_prefix"]) if e["query_prefix"] else (f'{e["query_task"]}|{e["doc_task"]}' if e.get("query_task") else "-"),
                 "Zeitstempel Encoding (UTC)": encode_timestamp(HARNESS / m)})
display(pd.DataFrame(rows).style.hide(axis="index"))
e0, en0 = NATIVE["e5-large"]["experiment"], NATIVE["e5-large"]["environment"]
print("Stack:", {k: e0[k] for k in ("python", "sentence_transformers", "torch", "transformers", "device")},
      "numpy", en0["numpy"], "| Hardware:", en0["platform"], f'{en0["cpu_count"]} CPU-Kerne')
api = NATIVE["openai-3-large"]["experiment"]["openai_api"]
print("OpenAI-Paket:", NATIVE["openai-3-large"]["experiment"]["openai"], "| Abruf 3-large:", api["first_request_utc"][:19], "bis", api["last_request_utc"][:19])

### Rohbefunde

In [ ]:
mrr = {k: float(cm.rr10(r).mean()) for k, r in ALL_RANKS.items()}
sig = conf.assign(Modell=conf.b.map(LABEL)).set_index("Modell")
print("MRR@10 (native Länge):", {LABEL[k]: round(v, 4) for k, v in mrr.items()})
print("Konfirmatorisch, p Holm < 0,05:", [i for i, r in sig.iterrows() if r.p_holm < 0.05], "| nicht unterscheidbar von BM25:", [i for i, r in sig.iterrows() if r.p_holm >= 0.05])
print("Unter der Deckeneffekt-Schwelle (konfirmatorisch):", [i for i, r in sig.iterrows() if r.not_informative] or "keiner")
print(f'BM25-de gegen BM25 Standard: MRR@10 {mrr["bm25_de"]:.4f} gegen {mrr["bm25"]:.4f}')

**Zusammenfassung.** Rohbefunde aus den gespeicherten Läufen (native Eingabelänge, GermanQuAD-Test, MRR@10, Seed 42) stehen in Tabelle 2 und 3 sowie im Block „Rohbefunde“ oben. Die konfirmatorische Gruppe umfasst sechs Tests (Modelle gegen BM25 Standard); Paarvergleiche, 512-Sensitivität, BM25-de und die drei explorativ getesteten Modelle sind explorativ. Die Konsistenzprüfung bestätigt, dass alle Kennzahlen aus den Per-Query-Rängen mit den drei Vergleichsdateien übereinstimmen. Grenzen: ein Datensatz (474 Passagen, je Frage genau eine relevante Passage), Deckeneffekte im oberen MRR-Bereich, CPU-Läufe ohne Wiederholung (Encoding ist deterministisch bis auf Hardware-Rundung).